In [ ]:
import numpy as np 
import pandas as pd 

from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.datasets import make_classification

In [3]:
X,y = make_classification(n_samples=10000, n_features=10,n_informative=3)

In [5]:
X_train,X_test,y_train,y_test = train_test_split(X,y,test_size=0.2,random_state=3)
X_train.shape

(8000, 10)

In [7]:
from sklearn.metrics import accuracy_score

In [8]:
dt = DecisionTreeClassifier()
dt.fit(X_train,y_train)
y_pred = dt.predict(X_test)
print("Acc:",accuracy_score(y_test,y_pred))

Acc: 0.9225


Bagging on Decision Trees

In [10]:
from sklearn.ensemble import BaggingClassifier

In [12]:
bgdt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25 ,# 25% of the data will be given to every model
    bootstrap= True ,# bagging will be done(rows can repeat)
    random_state= 42 
)
bgdt.fit(X_train,y_train)
y_pred = bgdt.predict(X_test)
print("Acc:",accuracy_score(y_test,y_pred))

Acc: 0.9575


In [14]:
bgdt.estimators_samples_[0].shape

(2000,)

In [15]:
bgdt.estimators_features_[0].shape

(10,)

ON SVM

In [13]:
svc = SVC()
svc.fit(X_train,y_train)
y_pred = svc.predict(X_test)
print("Acc:",accuracy_score(y_test,y_pred))

Acc: 0.945


In [17]:
bgsv= BaggingClassifier(
    estimator=SVC(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    random_state=42
)
     

bgsv.fit(X_train,y_train)
y_pred = bgsv.predict(X_test)
print("Bagging using SVM",accuracy_score(y_test,y_pred))

Bagging using SVM 0.9305


Pasting

In [19]:
bgpt = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=False,
    random_state=42,
    verbose = 1,
    n_jobs=-1
)
     

bgpt.fit(X_train,y_train)
y_pred = bgpt.predict(X_test)
print("Pasting classifier",accuracy_score(y_test,y_pred))

[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    8.9s remaining:   44.7s
[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    9.6s finished
[Parallel(n_jobs=12)]: Using backend LokyBackend with 12 concurrent workers.
[Parallel(n_jobs=12)]: Done   2 out of  12 | elapsed:    0.0s remaining:    0.3s


Pasting classifier 0.958


[Parallel(n_jobs=12)]: Done  12 out of  12 | elapsed:    0.2s finished


Random Subspaces

In [21]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=1.0,
    bootstrap=False,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)
     

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Subspaces classifier",accuracy_score(y_test,y_pred))

Random Subspaces classifier 0.9545


Random Patches

In [22]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    max_features=0.5,
    bootstrap_features=True,
    random_state=42
)
     

bag.fit(X_train,y_train)
y_pred = bag.predict(X_test)
print("Random Patches classifier",accuracy_score(y_test,y_pred))

Random Patches classifier 0.9395


OOB Score

In [24]:
bag = BaggingClassifier(
    estimator=DecisionTreeClassifier(),
    n_estimators=500,
    max_samples=0.25,
    bootstrap=True,
    oob_score=True,
    random_state=42
)
bag.fit(X_train,y_train)
bag.oob_score_

0.953

In [25]:
y_pred = bag.predict(X_test)
print("Accuracy",accuracy_score(y_test,y_pred))

Accuracy 0.9575


Applying Grid Search CV

In [ ]:
from sklearn.model_selection import GridSearchCV
parameters = {
    'n_estimators': [50,100,500], 
    'max_samples': [0.1,0.4,0.7,1.0],
    'bootstrap' : [True,False],
    'max_features' : [0.1,0.4,0.7,1.0]
    }
     

search = GridSearchCV(BaggingClassifier(), parameters, cv=5)
     

search.fit(X_train,y_train)